<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/STEP_2_FEATURE_ENGINEERING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 2: FEATURE ENGINEERING**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict

In [ ]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

# Date and Time Features

In [ ]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)

# Create a Product-Region Matrix and Network

In [ ]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

print(product_region.head(20))

In [ ]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [ ]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Create Shipping Mode Statistics

In [ ]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']


# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

# Define Multi-Region Stock

In [ ]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [ ]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))

# Merge the network

In [ ]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [ ]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))

 # Find candidate shipping regions

In [ ]:
candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv", encoding="latin1")

# Load the train, validation, and test dataframes
train_df = pd.read_csv("dataco_rl_train.csv", encoding="latin1")
val_df = pd.read_csv("dataco_rl_validation.csv", encoding="latin1")
test_df = pd.read_csv("dataco_rl_test.csv", encoding="latin1")

if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(columns={"Order Region": "Fulfillment_Region"})

# Keep ONE best fulfillment region per product to prevent row multiplication
candidate_regions = (
    candidate_regions
    .drop_duplicates(subset=["Product Category Id"], keep="first")
    [["Product Category Id", "Fulfillment_Region"]]
)

for name, df_obj in {
    "train_df": train_df,
    "val_df": val_df,
    "test_df": test_df
}.items():
    before = len(df_obj)

    merged = df_obj.merge(
        candidate_regions,
        on="Product Category Id",
        how="left"
    )

    after = len(merged)

    print(f"{name}: before={before}, after={after}")

    if after != before:
        raise ValueError(f"{name} row count changed during fulfillment-region merge.")

    if name == "train_df":
        train_df = merged
    elif name == "val_df":
        val_df = merged
    else:
        test_df = merged

# Save Outputs

In [ ]:
 network.to_csv(
   "product_region_network.csv",
    index=False, encoding="latin1"
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False, encoding="latin1"
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False, encoding="latin1"
)

print("Files created successfully")

# Shipping Engineering

In [ ]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [ ]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [ ]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [ ]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [ ]:
stocking = pd.read_csv(
    "product_stocking_scores.csv", encoding='latin1'
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [ ]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [ ]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [ ]:
df = df.copy()

df["Shipping Cost"] = df["Mode_Cost"]

df["Route"] = (
    df["Order Region"].astype(str)
    + " -> "
    + df["Order Country"].astype(str)
)

route_mode_cost = (
    df.groupby(["Route", "Shipping Mode"])["Shipping Cost"]
      .mean()
      .reset_index()
      .rename(columns={"Shipping Cost": "Route_Mode_Cost"})
)

df = df.merge(
    route_mode_cost,
    on=["Route", "Shipping Mode"],
    how="left"
)

# RL State Features

In [ ]:
state_features = [
    'Product Category Id',
    'stocking_score',
    'margin_pct',
    'shipping_delay',
    'regions_served',
    'profit_per_item'
]

# Reward Features

In [ ]:
df["reward"] = (
    15.0 * df["Order Profit Per Order"]
    - 1.0 * df["Days for shipping (real)"]
    - 2.0 * df["Late_delivery_risk"]
    - 0.02 * df["Route_Mode_Cost"]
)

# Save Engineered Dataset

In [ ]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False, encoding="latin1"
)

print(df.shape)